<a href="https://colab.research.google.com/github/E-Sentinel-Project/E-Sentinel/blob/master/MobiFall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import zipfile
import os

# Upload your MobiFall zip file
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print("Uploaded:", zip_name)


Saving archive (8).zip to archive (8).zip
Uploaded: archive (8).zip


In [ ]:
extract_path = "MobiFall"

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction done.")
print("Top-level folders:")
print(os.listdir(extract_path))


Extraction done.
Top-level folders:
['MobiFall_Dataset_v2.0']


In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
# ---------------- Kalman 1D ----------------
class Kalman1D:
    def __init__(self):
        self.x = 0.0
        self.p = 1.0
        self.q = 0.02
        self.r = 0.1

    def update(self, measurement):
        self.p += self.q
        k = self.p / (self.p + self.r)
        self.x += k * (measurement - self.x)
        self.p *= (1 - k)
        return self.x


# ---------------- Normalization Functions ----------------
def normalize(v, min_v, max_v):
    return np.clip((v - min_v) / (max_v - min_v), 0, 1)

def normalizeStableGravity(acc):
    return 1 - np.clip(abs(acc - 9.8) / 5, 0, 1)

def normalizeSmallJerk(j):
    return np.clip(1 - (j / 15), 0, 1)


In [ ]:
def detect_fall_neutrosophic(data, actual_label):
    kFilter = Kalman1D()
    last_acc = np.zeros(3)
    last_ts = None

    predictions = []
    ground_truth = []

    for i in range(len(data)):
        x, y, z, timestamp = data.iloc[i]

        rawAccel = np.sqrt(x*x + y*y + z*z)
        filteredAccel = kFilter.update(rawAccel)

        if last_ts is None:
            dt = 0
            jerk = 0
        else:
            dt = (timestamp - last_ts)
            diff = np.sqrt((x-last_acc[0])**2 +
                           (y-last_acc[1])**2 +
                           (z-last_acc[2])**2)
            jerk = diff/dt if dt > 0 else 0

        last_ts = timestamp
        last_acc = np.array([x,y,z])

        angle = np.degrees(np.arccos(np.clip(z/filteredAccel, -1, 1)))

        # ---------------- T ----------------
        T = (
            normalize(filteredAccel, 15, 30) * 0.55 +
            normalize(jerk, 10, 35) * 0.30 +
            normalize(angle, 35, 85) * 0.15
        )
        T = np.clip(T, 0, 1)

        # ---------------- I ----------------
        I = (
            normalize(jerk, 3, 12) * 0.4 +
            normalize(angle, 10, 35) * 0.6
        )
        I = np.clip(I, 0, 0.3)

        # ---------------- F ----------------
        F = (
            normalizeStableGravity(filteredAccel) * 0.6 +
            normalizeSmallJerk(jerk) * 0.4
        )
        F = np.clip(F, 0.1, 1)

        # -------- STRICT DECISION RULE --------
        predictedFall = (T - F > I) and (T > 0.6)

        predictions.append(1 if predictedFall else 0)
        ground_truth.append(actual_label)

    return predictions, ground_truth


In [ ]:
all_files = glob("MobiFall/**/*_acc_*.txt", recursive=True)

print("Total ACC files found:", len(all_files))


Total ACC files found: 630


In [ ]:
all_predictions = []
all_labels = []

for file in all_files:

    try:
        with open(file, 'r') as f:
            lines = f.readlines()

        # Find @DATA
        data_start = 0
        for i, line in enumerate(lines):
            if "@DATA" in line.upper():
                data_start = i + 1
                break

        numeric_lines = lines[data_start:]

        data = []
        for line in numeric_lines:
            parts = line.strip().split(",")
            if len(parts) >= 4:
                try:
                    timestamp = float(parts[0])
                    x = float(parts[1])
                    y = float(parts[2])
                    z = float(parts[3])
                    data.append([x, y, z, timestamp])
                except:
                    continue

        if len(data) == 0:
            continue

        df = pd.DataFrame(data, columns=['x','y','z','timestamp'])

        # Convert timestamp ns → seconds
        df['timestamp'] = df['timestamp'] / 1e9

        # Label
        if "FALLS" in file.upper():
            label = 1
        else:
            label = 0

        preds, gts = detect_fall_neutrosophic(df[['x','y','z','timestamp']], label)

        all_predictions.extend(preds)
        all_labels.extend(gts)

    except Exception as e:
        print("Error in file:", file)
        continue

print("Processing Complete.")
print("Total samples:", len(all_labels))


Processing Complete.
Total samples: 1044765


In [ ]:
from glob import glob

acc_files = glob("MobiFall/**/*_acc_*.txt", recursive=True)
print("Total ACC files:", len(acc_files))


Total ACC files: 630


In [ ]:
event_TP = 0
event_FP = 0
event_TN = 0
event_FN = 0

for file in acc_files:

    try:
        with open(file, 'r') as f:
            lines = f.readlines()

        # Find @DATA
        data_start = 0
        for i, line in enumerate(lines):
            if "@DATA" in line.upper():
                data_start = i + 1
                break

        numeric_lines = lines[data_start:]

        data = []
        for line in numeric_lines:
            parts = line.strip().split(",")
            if len(parts) >= 4:
                try:
                    timestamp = float(parts[0])
                    x = float(parts[1])
                    y = float(parts[2])
                    z = float(parts[3])
                    data.append([x, y, z, timestamp])
                except:
                    continue

        if len(data) == 0:
            continue

        df = pd.DataFrame(data, columns=['x','y','z','timestamp'])
        df['timestamp'] = df['timestamp'] / 1e9  # ns → seconds

        # Label
        if "FALLS" in file.upper():
            label = 1
        else:
            label = 0

        preds, _ = detect_fall_neutrosophic(df[['x','y','z','timestamp']], label)

        detected = any(preds)

        if label == 1:
            if detected:
                event_TP += 1
            else:
                event_FN += 1
        else:
            if detected:
                event_FP += 1
            else:
                event_TN += 1

    except:
        continue

print("Event Processing Complete.")


Event Processing Complete.


In [ ]:
print("Event-Level Confusion Matrix:")
print([[event_TN, event_FP],
       [event_FN, event_TP]])

sensitivity = event_TP / (event_TP + event_FN) if (event_TP + event_FN) > 0 else 0
specificity = event_TN / (event_TN + event_FP) if (event_TN + event_FP) > 0 else 0
f1 = (2 * event_TP) / (2 * event_TP + event_FP + event_FN) if (2 * event_TP + event_FP + event_FN) > 0 else 0

print("\nEvent-Level Metrics:")
print("Sensitivity:", round(sensitivity, 4))
print("Specificity:", round(specificity, 4))
print("F1-score:", round(f1, 4))


Event-Level Confusion Matrix:
[[255, 87], [81, 207]]

Event-Level Metrics:
Sensitivity: 0.7188
Specificity: 0.7456
F1-score: 0.7113
